In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import seaborn as sns

from climate_attitudes.datasets.reduced_no_imputation import schema
from climate_attitudes.visualisation import configure_mpl

RANDOM_SEED = 202606041925

rng = np.random.default_rng(RANDOM_SEED)


np.set_printoptions(linewidth=200)

configure_mpl(Path("../fonts"))

DATA_PATH = Path("../reports/thesis/results/data/model/bootstrapped_fit")

schema = schema.post_index()

In [ ]:
asym_params = np.load(DATA_PATH / "ising_no_structure.npz")["params"][:, 8:].reshape(
    (-1, 8, 8)
)

_sym_params = np.load(DATA_PATH / "sym_ising_no_structure.npz")["params"][:, 8:]
sym_params = np.zeros_like(asym_params)
for i, p in enumerate(_sym_params):
    sym_params[i][np.triu_indices(8)] = p

asym_params[np.abs(asym_params) < 1e-2] = 0
sym_params[np.abs(sym_params) < 1e-2] = 0

In [ ]:
def in_degree(params, spin_idx: int) -> npt.NDArray[np.int64]:
    return (~np.isclose(np.delete(params[..., spin_idx], spin_idx, axis=-1), 0.0)).sum(
        axis=-1
    )


def out_degree(params, spin_idx: int) -> npt.NDArray[np.int64]:
    return (~np.isclose(np.delete(params[:, spin_idx], spin_idx, axis=-1), 0.0)).sum(
        axis=-1
    )


def in_strength(params, spin_idx: int) -> npt.NDArray[np.float64]:
    return np.abs(np.delete(params[..., spin_idx], spin_idx, axis=-1)).sum(axis=-1)


def out_strength(params, spin_idx: int) -> npt.NDArray[np.float64]:
    return np.abs(np.delete(params[:, spin_idx], spin_idx, axis=-1)).sum(axis=-1)


def degree(params, spin_idx: int) -> npt.NDArray[np.int64]:
    return in_degree(params, spin_idx) + out_degree(params, spin_idx)


def strength(params, spin_idx: int) -> npt.NDArray[np.float64]:
    return in_strength(params, spin_idx) + out_strength(params, spin_idx)

In [ ]:
sym_degree_ccw = degree(sym_params, 2)
sym_strength_ccw = strength(sym_params, 2)

asym_indegree_ccw = in_degree(asym_params, 2)
asym_outdegree_ccw = out_degree(asym_params, 2)
asym_instrength_ccw = in_strength(asym_params, 2)
asym_outstrength_ccw = out_strength(asym_params, 2)

sym_degree_pol = degree(sym_params, 5)
sym_strength_pol = strength(sym_params, 5)

asym_indegree_pol = in_degree(asym_params, 5)
asym_outdegree_pol = out_degree(asym_params, 5)
asym_instrength_pol = in_strength(asym_params, 5)
asym_outstrength_pol = out_strength(asym_params, 5)

In [ ]:
fig, axes = plt.subplots(ncols=2, constrained_layout=True)

axes[0].hist(sym_degree_ccw, label="Degree (symmetric)")
axes[0].hist(asym_indegree_ccw, label="In-degree (asymmetric)")
axes[0].hist(asym_outdegree_ccw, label="Out-degree (asymmetric)")

axes[1].hist(sym_strength_ccw, label="Strength (symmetric)")
axes[1].hist(asym_instrength_ccw, label="In-strength (asymmetric)")
axes[1].hist(asym_outstrength_ccw, label="Out-strength (asymmetric)")

axes[0].legend()
axes[1].legend()

In [ ]:
np.percentile(asym_outdegree_pol - asym_indegree_pol, 5)

In [ ]:
fig, axes = plt.subplots(ncols=2, nrows=2, figsize=(8, 3.8), constrained_layout=True)


sns.histplot(
    asym_outstrength_pol - asym_instrength_pol,
    ax=axes[1, 0],
    stat="density",
    alpha=0.4,
    edgecolor=(1, 1, 1, 0.4),
)
sns.histplot(
    asym_outdegree_pol - asym_indegree_pol,
    ax=axes[1, 1],
    stat="density",
    alpha=0.4,
    edgecolor=(1, 1, 1, 0.4),
    bins=(-0.5, 0.5, 1.5, 2.5, 3.5, 4.5),
    shrink=0.5,
)
sns.histplot(
    asym_outstrength_ccw - asym_instrength_ccw,
    ax=axes[0, 0],
    stat="density",
    alpha=0.4,
    edgecolor=(1, 1, 1, 0.4),
)

axes[0, 1].set_xlim(-0.5, 4.5)
axes[0, 1].set_xticks(np.arange(5))

axes[0, 0].set_xlim(-0.1, 0.7)
axes[1, 0].set_xlim(-0.1, 0.7)

axes[0, 0].axvline(
    np.percentile(asym_outstrength_ccw - asym_instrength_ccw, 5),
    linestyle="dashed",
    color="k",
    linewidth=0.75,
    label="5% percentile",
)
axes[1, 0].axvline(
    np.percentile(asym_outstrength_pol - asym_instrength_pol, 5),
    linestyle="dashed",
    color="k",
    linewidth=0.75,
)
axes[1, 1].axvline(
    np.percentile(asym_outdegree_pol - asym_indegree_pol, 5),
    linestyle="dashed",
    color="k",
    linewidth=0.75,
)

axes[0, 1].spines.left.set_visible(False)
axes[0, 1].spines.bottom.set_visible(False)
axes[0, 1].tick_params("both", length=0)
axes[0, 1].set_xticks([])
axes[0, 1].set_yticks([])

for ax in axes.flatten():
    ax.set_ylabel(None)
    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)

fig.supylabel("Density", y=0.6, fontsize=15)

axes[0, 1].set_ylabel(
    "Climate worry", rotation=0, ha="left", va="center", labelpad=20, fontsize=15
)
axes[0, 1].yaxis.set_label_position("right")

axes[1, 1].set_ylabel(
    "Political views", rotation=0, ha="left", va="center", labelpad=20, fontsize=15
)
axes[1, 1].yaxis.set_label_position("right")

axes[1, 0].set_xlabel(r"$\Delta_\text{strength}$", fontsize=18)
axes[1, 1].set_xlabel(r"$\Delta_\text{degree}$", fontsize=18)

axes[1, 0].set_xlabel(
    "Excess outbound influence\n($\\Delta_\\text{strength centrality}$)", fontsize=16
)
axes[1, 1].set_xlabel(
    "Excess outbound connections\n($\\Delta_\\text{degree centrality}$)", fontsize=16
)

fig.legend(loc="lower center", bbox_to_anchor=(0.35, 1.0), frameon=False)

fig.savefig(
    "../reports/thesis/results/figures/model/centrality_diffs.svg",
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)

In [ ]:
fig, ax = plt.subplots(figsize=(2.55, 1.5), constrained_layout=True)

sns.histplot(
    asym_outstrength_ccw - asym_instrength_ccw,
    ax=ax,
    stat="density",
    alpha=0.4,
    edgecolor=(1, 1, 1, 0.4),
)

ax.axvline(
    np.percentile(asym_outstrength_ccw - asym_instrength_ccw, 5),
    linestyle="dashed",
    color="k",
    linewidth=0.75,
)
ax.set_xlabel("Strength centrality difference")

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(5.77, 2), constrained_layout=True)

sns.distplot(asym_outdegree_pol - asym_indegree_pol, ax=axes[0], kde=False)
sns.distplot(asym_outstrength_pol - asym_instrength_pol, ax=axes[1])

print(np.percentile(asym_outdegree_pol - asym_indegree_pol, 5))
print(np.percentile(asym_outstrength_pol - asym_instrength_pol, 5))

axes[0].set_title("Out-degree - in-degree")
axes[1].set_title("Out-strength - in-strength")

In [ ]:
asym_outdegree_pol

In [ ]:
asym_indegree_pol

In [ ]:
asym_params[0][:, 5]